# W1 · K3 — the release feed check, run by hand

**Checks:** the `Release feed` workflow (`release-feed-check.yml`) is green when run on demand, and the feed the app reads
(`releases/latest/download/latest.json`) looks right from *this* PC. A green run closes roadmap **K3**.

**You need:** the GitHub CLI `gh`, logged in (`gh auth login`) with permission to run workflows on `enjay27/resonance-stream`.
No Administrator needed, no game.

In [ ]:
import os, sys, time
from datetime import datetime, timezone
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "w1" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find test-w1 above it
from w1 import common, feed
from w1.common import Recorder, capture
rec = Recorder("w1-feed")
SLUG = "enjay27/resonance-stream"
WORKFLOW = "release-feed-check.yml"

## 1 · `gh` is installed and logged in

In [ ]:
code, out = capture(["gh", "auth", "status"], fixture="gh_auth_ok.txt")
rec.auto("K3-1", "gh is installed and logged in", code == 0, out.strip().splitlines()[0] if out.strip() else "")
if code != 0:
    raise SystemExit("Install gh and run `gh auth login`, then run again.")

## 2 · Run the workflow on demand and wait for it

In [ ]:
since = "2026-10-04T06:19:50Z" if common.dry_run() else datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
code, out = capture(["gh", "workflow", "run", WORKFLOW, "--repo", SLUG], fixture="gh_dispatch.txt")
rec.auto("K3-2", "workflow dispatched", code == 0, out.strip()[:200])
if code != 0:
    raise SystemExit("Could not start the workflow (permissions?). Run it from GitHub: Actions > Release feed > Run workflow.")

In [ ]:
RUNS = ["gh", "run", "list", "--workflow", WORKFLOW, "--repo", SLUG, "--event", "workflow_dispatch",
        "--json", "databaseId,status,conclusion,createdAt,event,url", "--limit", "5"]

def fetch():
    code, text = capture(RUNS, fixture=os.environ.get("W1_FIXTURE_RUNS", "gh_runs_success.json"))
    run = feed.pick_dispatched_run(text, since) if code == 0 else None
    return feed.run_state(run) if run else None

if common.dry_run():
    clock = feed.FakeClock(); sleep, now = clock.sleep, clock.now
else:
    sleep, now = time.sleep, time.time
state, url = feed.poll(fetch, sleep, now, timeout=600, interval=10)
rec.record("K3-3", "Release feed workflow run is green", state, url)

## 3 · The feed, read from this PC like the app reads it

In [ ]:
text = common.fetch_text(f"https://github.com/{SLUG}/releases/latest/download/latest.json", fixture="real_latest_json_v0.6.1.json")
for title, ok, evidence in feed.check_feed_json(text, SLUG):
    rec.auto("K3-4", title, ok, evidence)

## 4 · Key custody (K3b) — only you can do this

In [ ]:
rec.manual("K3b", "primary and backup signing keys and their passwords are stored in two places each",
           "Copy the primary key file + password and the backup key file + password to a second place "
           "(a second drive, a password manager). Losing both strands every 0.6.x copy.",
           "each of the four items exists in two places")

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())